# Collect AVH Data for Analysis
The following code brings together data generated and collected by Feng, Changye, and George into a single dataframe for future analysis. Right now the focus is on transcript level information for the sandwich analysis.

TODO

    1) How to handle NAs? Can implement smarter NA handling by only removing rows that have NAs in the columns we care about
    2) Log of WER
    3) Add in pause proportion
    4) Do not use SNR for WER

## Import libraries and functions

In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
# import statsmodels.api as sm
# from statsmodels.regression.mixed_linear_model import MixedLM
import numpy as np
from io import StringIO
from pathlib import Path

# directory_path = Path("another_new_directory")

# directory_path.mkdir(parents=True, exist_ok=True)
# print(f"Directory '{directory_path}' ensured to exist.")
out_folder = "/edata/obdw/sandwich_analysis_data/"


In [2]:
# Import shared utilities
import sys
import importlib
sys.path.insert(0, '/home/NETID/emd5')
import avh_utils
# Reload in case the module has been updated
importlib.reload(avh_utils)
from avh_utils import decode_variable_name, data_dictionary


In [3]:
def process_uid(series: pd.SparseDtype) -> pd.Series:
    """
    Extract unique pid from file names in a pandas Series. From Changye Li

    Args:
        series (pd.Series): Series containing file names of the transcripts

    Returns:
        pd.Series: Series with processed PIDs as integers
    """
    # Split the strings and get the first element (pid part)
    pids = series.str.split('-').str[0]
    
    # Clean up the pid strings
    pids = (pids.str.rstrip('@avh')
                .str.lstrip('u')
                .str.lstrip('0')
                .astype(int))
    
    return pids


## Read in Data

In [4]:
ptcpt_baseline_path = "/edata/changye/restore/avh-data/avh_record_baseline_kwargs.jsonl"# I wil use this file as the main
#this was old path
#ptcpt_baseline_path = "/edata/changye/avh-data/avh_record_baseline_kwargs.jsonl"# I wil use this file as the main

# rcd_whisperx_scores_path = "/edata/feng/avh/whisperx_baseline_with_time_wer_english_full.json"
# old path
rcd_whisperx_scores_path = "/edata/feng/avh/whisperx_merged_with_manual.json" #"/edata/feng/avh/whisperx_full_cleaned.json"
rcd_snr_path = "/edata/changye/restore/avh-new/data/snr.csv"
#rcd_snr_path = "/edata/changye/avh-new/data/snr.csv"
rcd_mos_path = "/edata/changye/restore/avh-data/pred_mos.csv"
rcd_coh_recording_level_path = "/edata/george/debias_merged_new.csv"




In [5]:
baseline_df = pd.read_json(ptcpt_baseline_path, lines=True)
baseline_df = baseline_df.set_index("pid", drop=True).rename_axis("file")
baseline_df = baseline_df.rename(columns={"uid":"pid"})

rcd_whisperx_scores_df = pd.read_json(rcd_whisperx_scores_path, lines=True)
rcd_whisperx_scores_df = rcd_whisperx_scores_df.set_index("file", drop=True)#.rename_axis("file")
# rcd_whisperx_scores_df["pid"] = process_uid(rcd_whisperx_scores_df.pid)


rcd_snr_df = pd.read_csv(rcd_snr_path)
rcd_snr_df = rcd_snr_df.set_index("file")
rcd_snr_df["pid"] = process_uid(rcd_snr_df.index)

rcd_mos_df = pd.read_csv(rcd_mos_path)
rcd_mos_df = pd.DataFrame(rcd_mos_df.groupby(['file_name'])['pred_mos'].mean())
rcd_mos_df["pid"] = process_uid(rcd_mos_df.index)


rcd_level_coh_df = pd.read_csv(rcd_coh_recording_level_path)# Similar to above but for recordings.
rcd_level_coh_df["file"] = rcd_level_coh_df.file.apply(lambda x: x.strip(".txt"))
rcd_level_coh_df = rcd_level_coh_df.set_index("file")
rcd_level_coh_df["pid"] = process_uid(rcd_level_coh_df.index)


## Clean Data

### Remove duplicated columns
Not all will be removed but this should clean it up some

In [6]:
def remove_duplicated_columns(main_df, secondary_df, extra_cols_to_drop):
    """Remove duplicate columns from secondary_df before merge."""
    intersection_columns = set(main_df).intersection(secondary_df).union(extra_cols_to_drop)
    updated_df = secondary_df.drop(intersection_columns, axis=1)
    return updated_df


In [7]:
coh_cols_to_drop = ['contact',
 'temper-outbursts',
 'feeling-blocked',
 'worrying-too-much',
 'easily-hurt',
 'feeling-watched',
 'feeling-tense',
 'heavy-feelings-in-arms-legs',
 'feeling-nervous',
 'feeling-lonely',
 'frequency',
 'bad-voices',
 'volume-of-voices',
 'voices-length',
 'interference-in-activities',
 'distressing-voices',
 'worthless-useless-voices',
 'clarity-of-voices',
 'follow-voices-orders',
 'time-of-day-of-voices',
 'social-situations',
 'where-are-the-voices',
 'typical-week',
 'if-no-explanation',
 'work-school-disruption',
 'social-leisure-disruption',
 'home-family-disruption',
 'school-work-missed',
 'less-productive-days',
#  'not-worked-for-other-reasons',# not removed since this is one George suggested to keep
 'little-interest-or-pleasure',
 'feeling-depressed',
 'trouble-sleeping',
 'feeling-tired',
 'appetite',
 'feeling-bad-about-self',
 'trouble-concentrating',
 'slow-fast-speaking',
 #'suicidal-thoughts',
 'impact-on-your-life', 'text']


In [8]:
rcd_level_coh_df = remove_duplicated_columns(main_df=baseline_df, secondary_df=rcd_level_coh_df, extra_cols_to_drop=coh_cols_to_drop)
rcd_whisperx_scores_df = remove_duplicated_columns(main_df=baseline_df, secondary_df=rcd_whisperx_scores_df, extra_cols_to_drop=["text"])
rcd_mos_df = remove_duplicated_columns(main_df=baseline_df, secondary_df=rcd_mos_df, extra_cols_to_drop=[])
rcd_snr_df = remove_duplicated_columns(main_df=baseline_df, secondary_df=rcd_snr_df, extra_cols_to_drop=[])


## Merge Data

In [9]:
main_analysis_df = pd.concat([baseline_df, rcd_whisperx_scores_df, rcd_snr_df, rcd_mos_df, rcd_level_coh_df], axis=1, join="inner")
assert main_analysis_df.shape[0] == len(file_names_intersection)
#print(main_analysis_df.columns)
#print(main_analysis_df.columns.tolist())


NameError: name 'file_names_intersection' is not defined

## Remove NA
I am ignoring certain columns I do not expect to use. Like many of the coherence columns. Otherwise we would be dropping many columns with NAs that we would never use.

In [ ]:
columns2ignore_na = ["not-worked-for-other-reasons"]
for column_name in main_analysis_df.columns:
    if "Coherence" in column_name and column_name != "sentCoherenceSentBertCumulativeCentroid":# the one we care about is sentCoherenceSentBertCumulativeCentroid
        columns2ignore_na.append(column_name)
# Code below just to look at columns with na
#  for x, y in zip(main_analysis_df.columns, list(main_analysis_df.isna().sum())):
#     if x in columns2ignore_na:
#         continue
#     print(x, y)
main_analysis_df.drop(columns2ignore_na, axis=1, inplace=True)
main_analysis_df.dropna(axis=0, inplace=True)
# main_analysis_df


In [ ]:
main_analysis_df.to_csv(out_folder + "main_merged_sandwich_analysis_data.csv")


## Identify Features for Analysis
The next step is to identify subsets of features to use for sandwich analysis. Listing the features and response variable out should not be hard for each analysis. Removing nan values will depend more on the response variable I believe, since nan categorical variables can be coded up in one-hot encoding. However it might be wise to remove rows with any nan values.

### Options for final cleaning

1) Remove nan values from each dataframe on a per analysis criteria
2) Remove nan values from the overall dataframe so each analysis is working with the same dataframe.
3) A hybrid approach of the above where rows are removed based off of nan values in the features from the overall dataframe and then nan values are removed in the response variable on a per analysis criteria.

## Feature Engineering
As of now, this is only one-hot encoding code I have from the previous analysis

In [ ]:
binmappers = {
    "education": {
        2.0: 1.0,
        3.0: 1.0,
        4.0: 1.0,
        5.0: 2.0,
        6.0: 2.0,
        7.0: 3.0,
        8.0: 3.0,
        np.nan: np.nan,
    }
}

def bin_age(age):
    """Bin age into coarse categories."""
    if np.isnan(age):
        age_bin = 0.0
    elif age == 999.0:
        age_bin = 0.0
    elif age < 30.0:
        age_bin = 1.0
    elif age < 45.0:
        age_bin = 2.0
    elif age < 65.0:
        age_bin = 3.0
    else:
        age_bin = 4.0
    return age_bin

def replaced_age_with_binning(df):
    """Replace raw age with binned_age."""
    temp_df = df.copy(deep=True)
    temp_df["binned_age"] = temp_df.age.apply(lambda x: bin_age(x))
    temp_df.drop("age", axis=1, inplace=True)
    return temp_df

def bin_substance_use(data_df):
    """Create opioid/opiates and any-drug-use binary flags."""
    data_copy_df = data_df.copy(deep=True)
    all_types_drug_use = ["opioids","marijuana","alcohol","steroids","cocaine","heroin","meth","ketamine","acid","bath-salts"]
    substances_of_interest = ["opioids","heroin"]

    data_copy_df.loc[:, all_types_drug_use] = pd.DataFrame(data_copy_df.loc[:, all_types_drug_use] == 1, dtype=np.float64)
    opiods_opiates_1_use = pd.Series(data_copy_df.loc[:, substances_of_interest].sum(axis=1) > 0, dtype=np.float64)
    all_types_drug_use_flag = pd.Series(data_copy_df.loc[:, all_types_drug_use].sum(axis=1) > 0, dtype=np.float64)

    data_copy_df.drop(labels=substances_of_interest, axis=1, inplace=True)
    data_copy_df["opioids-opiates"] = opiods_opiates_1_use
    data_copy_df["all_types_drug_use"] = all_types_drug_use_flag
    return data_copy_df

def one_hot_encode(df, cols2encode, missing_label="__MISSING__"):
    """One-hot encode cols2encode while dropping the least-frequent class per column."""
    df_enc = df[cols2encode].fillna(missing_label).astype(str)
    drop_values = [str(df_enc[col].value_counts().idxmin()) for col in cols2encode]

    try:
        enc = OneHotEncoder(drop=drop_values, sparse=False, handle_unknown="ignore", dtype=int)
    except TypeError:
        enc = OneHotEncoder(drop=drop_values, sparse_output=False, handle_unknown="ignore", dtype=int)

    enc.fit(df_enc)
    one_hot_encoded_df = pd.DataFrame(
        enc.transform(df_enc),
        columns=enc.get_feature_names_out(),
        index=df.index
    )
    return one_hot_encoded_df


### Create minimum analysis data: Basic
with race, age, and gender.

In [ ]:
cols2encode=["race", "gender", "binned_age"]
# X_temp = X_temp.replace(binmappers)# Mapping education to higher level categories
# X_temp.rename(columns={"education":"education_binned"},inplace=True)
minimum_analysis_df = replaced_age_with_binning(main_analysis_df)
minimum_analysis_df = minimum_analysis_df.loc[:,cols2encode + ["pid"]]

cols2encode=["race", "gender", "binned_age"]
X_one_hot = one_hot_encode(df=minimum_analysis_df, cols2encode=cols2encode)
X = minimum_analysis_df.drop(cols2encode, axis=1)
X_min_analysis = pd.concat([X, X_one_hot, Y_WER, Y_COH], axis=1, join="inner", ignore_index=False)
X_min_analysis.to_csv(out_folder + "basic_analysis.csv")
#X_min_analysis


### Create analysis data: Basic+
Everything in Basic with MOS and SNR added

In [ ]:
X_basic_plus = pd.concat([X_min_analysis, main_analysis_df.loc[:, ["snr", "pred_mos", "pause_proportion"]]], axis=1, join="inner", ignore_index=False)
assert X_basic_plus.shape[0] == X_min_analysis.shape[0], "Seem to have lost some columns during the join"
X_basic_plus.to_csv(out_folder + "basic_plus_analysis.csv")
#X_basic_plus


### Create analysis data: Basic+ Clinical
Everything in Basic+ with the additions of phq9 total, hpsvq total, scl global average

In [ ]:
clinical_base = ["phq9-total", "hpsvq-total-score", "scl-avg-global-score",
    'sds-1-work-school-disruption', 'sds-2-social-leisure-disruption', 'sds-3-home-family-disruption',
    'sds-4-school-work-missed', 'sds-5-less-productive-days', 'sds-6-2t-worked-for-other-reasons'
    ]
suicidal_candidates = [
    "phq9-9-suicidal-thoughts",
    "suicidal-thoughts",
    "phq9-suicidal-thoughts",
]

diagnosis_candidates = [
    "dx-alzheimers-parkinsons",
    "dx-bipolar-disorder",
    "dx-depression",
    "dx-tbi",
    "dx-migraines",
    "dx-schizoaffective-disorder",
    "dx-schizophrenia",
    "dx-ptsd",
    "dx-substance-use",
    "dx-seizures",
    "dx-none-of-the-above",
]

suicidal_col = next((c for c in suicidal_candidates if c in main_analysis_df.columns), None)
clinical_cols = [c for c in clinical_base if c in main_analysis_df.columns]
if suicidal_col is not None:
    clinical_cols.append(suicidal_col)
clinical_df = main_analysis_df.loc[:, list(dict.fromkeys(clinical_cols))].copy()

dx_groups = {
    "dx_group_smi": ["dx-bipolar-disorder", "dx-schizoaffective-disorder", "dx-schizophrenia", "dx-depression"],
    "dx_group_substance": ["dx-substance-use"],
    "dx_group_ptsd": ["dx-ptsd"],
    "dx_group_neuro_med": ["dx-alzheimers-parkinsons", "dx-migraines", "dx-seizures", "dx-tbi"],
}

def _to_binary_dx(s: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(s):
        return (s.fillna(0) > 0).astype(float)
    s2 = s.astype(str).str.strip().str.lower()
    true_vals = {"1", "1.0", "true", "t", "yes", "y"}
    return s2.isin(true_vals).astype(float)

dx_group_df = pd.DataFrame(index=main_analysis_df.index)
for group_name, cols in dx_groups.items():
    existing = [c for c in cols if c in main_analysis_df.columns]
    if existing:
        bin_df = pd.concat([_to_binary_dx(main_analysis_df[c]) for c in existing], axis=1)
        dx_group_df[group_name] = bin_df.max(axis=1).astype(float)
    else:
        dx_group_df[group_name] = 0.0

if "dx-none-of-the-above" in main_analysis_df.columns:
    dx_group_df["dx_group_none"] = _to_binary_dx(main_analysis_df["dx-none-of-the-above"])

X_basic_plus_clin = pd.concat([X_basic_plus, clinical_df, dx_group_df], axis=1, join="inner", ignore_index=False)
X_basic_plus_clin = X_basic_plus_clin.loc[:, ~X_basic_plus_clin.columns.duplicated()]

assert X_basic_plus_clin.shape[0] == X_basic_plus.shape[0], "Seem to have lost some rows during the join"
if suicidal_col is not None:
    assert suicidal_col in X_basic_plus_clin.columns

sds_total_components = [
    'sds-1-work-school-disruption',
    'sds-2-social-leisure-disruption',
    'sds-3-home-family-disruption'
 ]
available_sds_components = [c for c in sds_total_components if c in X_basic_plus_clin.columns]
if available_sds_components:
    X_basic_plus_clin["SDS_Total"] = X_basic_plus_clin[available_sds_components].sum(axis=1, min_count=1)

X_basic_plus_clin["PHQ9_high"] = (X_basic_plus_clin["phq9-total"] >= 10).astype(float)
X_basic_plus_clin["SDS_High"] = (X_basic_plus_clin["SDS_Total"] >= 21).astype(float) if "SDS_Total" in X_basic_plus_clin.columns else np.nan
X_basic_plus_clin.to_csv(out_folder + "basic_plus_clinical_analysis.csv")


### Create analysis data: Basic+ Clinical and SDH
Everything in Basic+ Clinical model with the additions of SDH features (sexuality, employment-status, education, and all drug data)

In [ ]:
# Get whether the user has used any drugs, and whether they have used opioids or opiates. This is important to include in the analysis since substance use is a key factor that can impact both WER and Coherence, and we want to make sure we are accounting for it in our models.
opiod_data = bin_substance_use(data_df=main_analysis_df).loc[:, ["opioids-opiates", "all_types_drug_use"]]
temp = main_analysis_df.copy()
temp = temp.merge(opiod_data, left_index=True, right_index=True, how='left')
temp = temp.replace(binmappers)# Mapping education to higher level categories
temp.rename(columns={"education":"education_binned"},inplace=True)
# Change which columns to drop in one-hot encoding. One hot encoding will create new columns for each category, so we want to make sure we are encoding the right ones. We want to encode the original categorical variables, not the binned or processed ones.
# Out of every categorical variable, one of the categories is dropped
cols2encode=["sexuality", "employment-status", "education_binned", "opioids-opiates", "all_types_drug_use"]
X_one_hot = one_hot_encode(df=temp, cols2encode=cols2encode)
X_basic_plus_clin_sdh = pd.concat([X_basic_plus_clin, X_one_hot], axis=1, join="inner", ignore_index=False)

assert X_basic_plus_clin_sdh.shape[0] == X_basic_plus_clin.shape[0], "Seem to have lost some columns during the join"
X_basic_plus_clin_sdh.to_csv(out_folder + "basic_plus_clinical_sdh_analysis.csv")
#X_basic_plus_clin_sdh


## Privacy-Safe Location Merge and Stratified Regression
This section removes exploratory display code and only reports regression coefficients and variable counts.


In [ ]:
location_path = "/home/NETID/obdw4/location_data_patrick/olivermergedcounts.csv"
location_df = pd.read_csv(location_path)
if "is_primary_cluster" in location_df.columns:
    location_df = location_df[location_df["is_primary_cluster"] == True].copy()

essential_columns = [
    "account_id", "RPL_THEMES", "RPL_THEME1", "RPL_THEME2", "RPL_THEME3", "RPL_THEME4",
    "PrimaryRUCA", "doctor_count", "pharmacy_count", "hospital_count",
    "park_count", "bus_station_count", "supermarket_count", "cluster"
]
available_columns = [c for c in essential_columns if c in location_df.columns]
location_df = location_df[available_columns].copy()

merged_df = X_basic_plus_clin_sdh.merge(location_df, left_on="pid", right_on="account_id", how="left")
X_location_clean = merged_df.drop(columns=["account_id"], errors="ignore").drop_duplicates()
single_value_cols = [col for col in X_location_clean.columns if X_location_clean[col].nunique(dropna=False) <= 1]
X_location_clean = X_location_clean.drop(columns=single_value_cols, errors="ignore")

categorical_location_cols = [c for c in ["PrimaryRUCA", "SecondaryRUCA", "STATEFP"] if c in X_location_clean.columns]
if categorical_location_cols:
    X_location_encoded = one_hot_encode(df=X_location_clean, cols2encode=categorical_location_cols)
    X_location_final = pd.concat([X_location_clean.drop(columns=categorical_location_cols), X_location_encoded], axis=1)
else:
    X_location_final = X_location_clean

X_location_final.to_csv(out_folder + "X_basic_plus_clin_sdh_location_encoded.csv")


In [ ]:
import re
X_stratified = X_location_final.copy()

if "PrimaryRUCA" in merged_df.columns:
    def classify_urban_rural(ruca_code):
        if pd.isna(ruca_code):
            return "Unknown"
        m = re.search(r"(-?\d+(?:\.\d+)?)", str(ruca_code).strip())
        if not m:
            return "Unknown"
        val = float(m.group(1))
        if val <= 1:
            return "UrbanCore"
        if val <= 3:
            return "Suburban"
        if val <= 6:
            return "LargeRural"
        if val <= 10:
            return "SmallTownRural"
        return "Rural"

    X_stratified["urban_rural_category"] = merged_df["PrimaryRUCA"].apply(classify_urban_rural)
    dummies = pd.get_dummies(X_stratified["urban_rural_category"], prefix="urban_rural", dummy_na=False)
    X_stratified = pd.concat([X_stratified, dummies], axis=1)

if "RPL_THEMES" in X_location_final.columns:
    X_stratified["high_svi"] = (X_location_final["RPL_THEMES"] >= 0.66).astype(float)

X_stratified = X_stratified.drop(columns=[
    "sds-1-work-school-disruption", "sds-2-social-leisure-disruption", "sds-3-home-family-disruption",
    "sds-4-school-work-missed", "sds-5-less-productive-days", "sds-6-2t-worked-for-other-reasons"
], errors="ignore")

X_stratified.to_csv(out_folder + "X_basic_plus_clin_sdh_location_stratified.csv")


In [ ]:
# Regression output restricted to coefficients and variable counts
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf

X_stratified = pd.read_csv(out_folder + "X_basic_plus_clin_sdh_location_stratified.csv", index_col=0)
outcomes = ["log_wer", "sentCoherenceSentBertCumulativeCentroid"]

for outcome_name in outcomes:
    if outcome_name not in X_stratified.columns:
        continue

    exclude = {"log_wer", "sentCoherenceSentBertCumulativeCentroid", "wer"}
    predictors = [c for c in X_stratified.columns if c not in exclude]
    terms = []
    for c in predictors:
        qc = f'Q("{c}")'
        if pd.api.types.is_object_dtype(X_stratified[c]) or isinstance(X_stratified[c].dtype, pd.CategoricalDtype):
            terms.append(f"C({qc})")
        else:
            terms.append(qc)

    formula = f'Q("{outcome_name}") ~ '+' + '.join(terms)
    model = smf.ols(formula=formula, data=X_stratified).fit(
        cov_type="cluster",
        cov_kwds={"groups": X_stratified["pid"], "use_correction": True}
    )

    coef_table = pd.DataFrame({
        "Variable": model.params.index,
        "Coefficient": model.params.values,
        "Std_Error": model.bse.values,
        "p_value": model.pvalues.values
    })

    mask = ~coef_table["Variable"].isin(["Intercept", "const"])
    reject_fdr, p_fdr, _, _ = multipletests(coef_table.loc[mask, "p_value"], alpha=0.05, method="fdr_bh")
    coef_table.loc[mask, "p_fdr"] = p_fdr
    coef_table.loc[mask, "fdr_significant"] = reject_fdr
    coef_table.loc[~mask, "p_fdr"] = np.nan
    coef_table.loc[~mask, "fdr_significant"] = False

    coef_table.to_csv(out_folder + f"regression_stratified_{outcome_name}.csv", index=False)

    print(f"Outcome: {outcome_name}")
    print(f"Variable count used in model: {len(predictors)}")
    print(coef_table[["Variable", "Coefficient", "Std_Error", "p_value", "p_fdr", "fdr_significant"]])
    print("-" * 80)
